# تقييم خطوة الترجمة على المحاضرة (Colab)

يقيس الخط النهائي `final_v1` وخط الأساس (Qwen3-4B، نموذج التشغيل الأول) وباقي الأنظمة المحفوظة **بالبوابة نفسها**:
نفس نوافذ الوقت، ونفس MetricX-24، ونفس فحص الإملاء.

- البيانات تبقى على Google Drive ولا تُرفع إلى أي مكان.
- النتائج تُكتب في `translation_step/evaluations/<eval_id>/auto.json` وتُضاف سطرًا في `evaluations/registry.jsonl`.
- التشغيل: Runtime → Change runtime type → **T4 GPU**، ثم شغّل الخلايا بالترتيب.

In [ ]:
# 1) Drive + code (branch below) + requirements
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

BRANCH = 'main'
LESSON = 'lesson_bfe775aa22957d44_academic_v2'
SUB = Path('outputs') / 'academic_fresh_pipeline' / LESSON
# Own folder, a shortcut in My Drive, the hidden shortcut-target folder Colab uses, or a shared drive.
_roots = [Path('/content/drive/MyDrive/ArabicDubbingV2')]
_roots += sorted(Path('/content/drive/.shortcut-targets-by-id').glob('*/ArabicDubbingV2'))
_roots += sorted(Path('/content/drive').glob('*/ArabicDubbingV2'))
_roots += sorted(Path('/content/drive').glob('*/*/ArabicDubbingV2'))
RUN_DIR = next((r / SUB for r in _roots if (r / SUB).exists()), None)
if RUN_DIR is None:
    print('Drive mounted, but ArabicDubbingV2 is not visible. What this account sees:')
    print('  MyDrive:', sorted(p.name for p in Path('/content/drive/MyDrive').iterdir())[:40])
    _sc = Path('/content/drive/.shortcut-targets-by-id')
    print('  shortcut targets:', sorted(str(p.relative_to(_sc)) for p in _sc.glob('*/*'))[:20] if _sc.exists() else 'none')
    raise FileNotFoundError('Add a shortcut: drive.google.com -> Shared with me -> right-click ArabicDubbingV2 -> '
                            'Organize -> Add shortcut -> My Drive (in the SAME account Colab mounted), then rerun.')
_probe = RUN_DIR / 'translation_step' / '.write_test'
try:
    _probe.write_text('ok'); _probe.unlink()
except OSError:
    raise PermissionError('Drive folder is read-only for this account: share ArabicDubbingV2 as Editor.')
print('RUN_DIR =', RUN_DIR)

TR_REPO = Path('/content/translation_step_repo')
if TR_REPO.exists():
    subprocess.run(['git', '-C', str(TR_REPO), 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(TR_REPO), 'checkout', '-q', '-B', BRANCH, 'origin/' + BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    'https://github.com/HamzaAlSamman/AI-Dubbing-Project.git', str(TR_REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sacrebleu', 'sentencepiece', 'num2words'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'hunspell', 'hunspell-ar'], check=True, capture_output=True)
sys.path.insert(0, str(TR_REPO / 'translation'))
print(subprocess.run(['git', '-C', str(TR_REPO), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# 2) evaluation (5-15 min: MetricX scores only texts it has not scored before)
from dubbing_translation.colab import evaluate_lecture
summary = evaluate_lecture(RUN_DIR, repo_dir=TR_REPO)